## Execution scope
The investigator selected five imputations fitted once on **complete development**, without outcomes, before ML. This notebook reuses the completed datasets from notebook 00 for every CV fold and fits no imputer. Holdout rows are filtered out before predictor encoding or model fitting. The original cohort, participant split, two CV schemes, five imputations, hyperparameter ranges and both Optuna budgets are retained.
The prior readmission run fitted imputers within CV folds. Its models and scores are preserved as a superseded strategy and are not reused for this run. An archive of its notebook, HTML and execution record is available in the project documentation.
Approved corrections remain: training-only Cox/Breslow **net survival**, arithmetic mean Brier at 3, 6, 12, 36 and 60 months, and positive zero-time representation solely at the XGBoost signed-label interface. Immediate readmissions, original analytical times and participants are unchanged.
**Retained limitation of internal CV.** The five missRanger imputations were fitted once on the complete development sample, without outcome or follow-up variables, before survival-model cross-validation. Predictors from each internal validation fold therefore contributed to learning the imputer, although that fold is excluded from fitting its survival model. This CV does not independently validate the complete preprocessing-and-model-selection workflow. missRanger OOB diagnostics assess the imputation forests and do not remove this dependence. The direction and magnitude of its effect on the retained workflow have not been isolated by the available comparisons. The investigator explicitly retained this scope on 2026-10-05.
**Separate holdout boundary.** Holdout rows and outcomes were excluded from fitting the development imputers, ML tuning and SHAP screening. However, notebook 00's descriptive early-death audit inspected holdout outcomes, and inherited discovery decisions remain documented. Do not describe the entire historical and reconstructed workflow as wholly independent of the holdout.
Configuration explicitly selects preflight, a timing pilot or full tuning. Preflight verifies all five completed inputs and all fold/imputation encodings without fitting models. The investigator authorized adaptation and execution of notebooks 01 and 02 on 2026-10-03.
The investigator reported computational constraints as the original motivation for the horizon-based Brier criterion. No historical timing comparison was available. Averaging instead of integrating the same five calculated values does not itself materially reduce computation.

In [1]:
#| label: readm-config-and-environment
#| message: false
import time
t0 = time.perf_counter()
# Resolve all machine and input paths from the single project configuration.
import os, sys, json, platform, importlib.metadata, gc, hashlib, pickle, functools, datetime, html
from pathlib import Path
root = Path.cwd().resolve()
while not (root / "config.yml").is_file():
    if root.parent == root:
        raise FileNotFoundError("config.yml was not found.")
    root = root.parent
import yaml
config = yaml.safe_load((root / "config.yml").read_text(encoding="utf-8"))
settings = config["xgboost_readmission"]
threads = int(config["imputation"]["threads"])
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = str(threads)
def resolve_path(value):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (root / path).resolve()
def source_path(name):
    return resolve_path(config["sources"][name])
assert Path(sys.prefix).name == config["runtime"]["xgboost_environment"], "Select the configured surv kernel."
mode = settings["mode"]
assert mode in ("preflight", "pilot", "full"), "Invalid execution mode."
# Import the installed scientific stack after limiting native thread pools.
import numpy as np
import pandas as pd
import pyarrow as pa
pa.set_cpu_count(threads)
pa.set_io_thread_count(threads)
import psutil
import xgboost as xgb
from sksurv.linear_model.coxph import BreslowEstimator
from sksurv.metrics import brier_score, concordance_index_ipcw
from sksurv.util import Surv
from sksurv.nonparametric import CensoringDistributionEstimator
from sklearn.model_selection import StratifiedKFold
import optuna
from threadpoolctl import threadpool_limits, threadpool_info
from IPython.display import HTML, display
thread_limits = threadpool_limits(limits=threads)
seed = int(config["imputation"]["seed"])
np.random.seed(seed)
horizons = np.asarray(settings["horizons_months"], dtype=float)
assert settings["brier_aggregation"] == "arithmetic_mean"
assert settings["probability_target"] == "net_survival_breslow"
assert threads == 2 and np.all(np.diff(horizons) > 0)
written = []
run_started = time.perf_counter()
process = psutil.Process()
start_rss_gib = process.memory_info().rss / 2**30
# Keep exploratory tables scrollable without adding a widget dependency.
def show_table(data, caption):
    safe_caption = html.escape(caption)
    table = data.to_html(index=False, escape=True, float_format=lambda x: f"{x:.6f}")
    display(HTML(f'<div role="region" tabindex="0" aria-label="{safe_caption}" style="max-height:420px;overflow:auto;max-width:100%;border:1px solid #ddd;padding:0.4rem"><strong>{safe_caption}</strong>{table}</div>'))
show_table(pd.DataFrame({"Component": ["Kernel", "Seed", "Native threads", "Available RAM (GiB)", "Brier aggregation", "Probability target"], "Value": [f"Python {platform.python_version()} / {Path(sys.prefix).name}", seed, threads, round(psutil.virtual_memory().available / 2**30, 3), settings["brier_aggregation"], settings["probability_target"]]}), "Execution contract")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

C:\Users\andre\miniconda3\envs\surv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Component,Value
Kernel,Python 3.11.14 / surv
Seed,2125
Native threads,2
Available RAM (GiB),54.299000
Brier aggregation,arithmetic_mean
Probability target,net_survival_breslow


Elapsed: 0.023 minutes


In [2]:
#| label: readm-predict-net-survival
#| message: false
t0 = time.perf_counter()
# Estimate the Cox baseline from training outcomes and explicit XGBoost log-risk margins.
def predict_net_survival(y_train, margin_train, margin_valid, times):
    margin_train = np.asarray(margin_train, dtype=float)
    margin_valid = np.asarray(margin_valid, dtype=float)
    times = np.asarray(times, dtype=float)
    if margin_train.shape != (len(y_train),) or margin_valid.ndim != 1:
        raise ValueError("Training and validation margins must be one-dimensional and aligned.")
    if times.ndim != 1 or not len(times) or (np.diff(times) <= 0).any() or (times < 0).any():
        raise ValueError("Evaluation times must be nonnegative, ordered and unique.")
    if not all(np.isfinite(a).all() for a in (margin_train, margin_valid, times, y_train["time"])):
        raise ValueError("Nonfinite margins or follow-up times.")
    if not y_train["event"].any() or (y_train["time"] < 0).any() or times[-1] > y_train["time"].max():
        raise ValueError("Invalid training follow-up or unsupported evaluation horizon.")
    shift = margin_train.mean()
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        baseline = BreslowEstimator().fit(margin_train - shift, y_train["event"], y_train["time"])
        indices = np.searchsorted(baseline.unique_times_, times, side="right")
        h0 = np.r_[0.0, baseline.cum_baseline_hazard_.y][indices]
        survival = np.exp(-np.exp(margin_valid - shift)[:, None] * h0[None, :])
    if not np.isfinite(survival).all() or ((survival < 0) | (survival > 1)).any():
        raise ValueError("Invalid predicted net survival.")
    if (np.diff(survival, axis=1) > 1e-12).any():
        raise ValueError("Predicted survival increases over time.")
    return survival
# Preserve the existing scorer and arithmetic aggregation without silent fallback values.
def mean_horizon_brier(y_train, y_valid, survival, times):
    times = np.asarray(times, dtype=float)
    if survival.shape != (len(y_valid), len(times)):
        raise ValueError("Survival predictions and validation outcomes are not aligned.")
    if not np.isfinite(survival).all() or ((survival < 0) | (survival > 1)).any():
        raise ValueError("Brier scoring requires finite probabilities between zero and one.")
    evaluated_times, scores = brier_score(y_train, y_valid, survival, times)
    if not np.array_equal(evaluated_times, times) or not np.isfinite(scores).all():
        raise ValueError("A requested Brier horizon could not be evaluated.")
    return float(np.mean(scores)), scores
# Preserve the original unweighted distance after Pareto selection.
def distance_to_ideal(c_index, mean_brier):
    c_index, mean_brier = np.asarray(c_index, dtype=float), np.asarray(mean_brier, dtype=float)
    if not np.isfinite(c_index).all() or not np.isfinite(mean_brier).all():
        raise ValueError("Selection metrics must be finite.")
    return np.sqrt((1 - c_index)**2 + mean_brier**2)
print("Net survival and mean Brier functions ready; no cohort model fitted.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Net survival and mean Brier functions ready; no cohort model fitted.
Elapsed: 0.000 minutes


In [3]:
#| label: readm-encode-zero-time-labels
#| message: false
t0 = time.perf_counter()
# Preserve events at time zero without altering the analytical follow-up times or shared ties.
def xgboost_cox_labels(y):
    times = np.asarray(y["time"], dtype=np.float32)
    events = np.asarray(y["event"], dtype=bool)
    epsilon = np.float32(settings["zero_time_label_months"])
    if not np.isfinite(times).all() or (times < 0).any() or not np.isfinite(epsilon) or epsilon <= 0:
        raise ValueError("XGBoost label times must be finite and nonnegative.")
    positive = times[times > 0]
    if len(positive) and epsilon >= positive.min():
        raise ValueError("The zero-time representation must precede every positive follow-up time.")
    represented = np.where(times == 0, epsilon, times)
    labels = np.where(events, represented, -represented).astype(np.float32)
    assert np.array_equal(labels > 0, events)
    assert np.array_equal(np.abs(labels[times > 0]), times[times > 0])
    return labels
print("Zero-time encoding changes only the XGBoost interface, for both events and censoring.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Zero-time encoding changes only the XGBoost interface, for both events and censoring.
Elapsed: 0.000 minutes


In [4]:
#| label: readm-synthetic-probability-controls
#| message: false
t0 = time.perf_counter()
# Check Breslow probabilities against an independent hand calculation with tied times.
synthetic_y = Surv.from_arrays([True, True, False, True], [1., 2., 2., 4.])
synthetic_risk = np.array([1., 2., 3., 4.])
synthetic_margin = np.log(synthetic_risk)
synthetic_valid_margin = np.log([0.1, 1., 80.])
synthetic_times = np.array([0., 1., 2., 3., 4.])
manual_hazard = np.array([0., 1 / 10, 1 / 10 + 1 / 9, 1 / 10 + 1 / 9, 1 / 10 + 1 / 9 + 1 / 4])
manual_survival = np.exp(-np.exp(synthetic_valid_margin)[:, None] * manual_hazard)
checked_survival = predict_net_survival(synthetic_y, synthetic_margin, synthetic_valid_margin, synthetic_times)
assert np.allclose(checked_survival, manual_survival, atol=1e-14, rtol=1e-12)
assert np.allclose(checked_survival, predict_net_survival(synthetic_y, synthetic_margin + 100, synthetic_valid_margin + 100, synthetic_times), atol=1e-14, rtol=1e-12)
# Without censoring, Brier must equal direct binary-status mean squared error.
uncensored_y = Surv.from_arrays([True] * 4, [1., 2., 4., 6.])
uncensored_times = np.array([1., 3., 5.])
uncensored_survival = predict_net_survival(uncensored_y, synthetic_margin, synthetic_margin, uncensored_times)
checked_mean, checked_scores = mean_horizon_brier(uncensored_y, uncensored_y, uncensored_survival, uncensored_times)
direct_scores = np.mean(((uncensored_y["time"][:, None] > uncensored_times).astype(float) - uncensored_survival)**2, axis=0)
assert np.allclose(checked_scores, direct_scores, atol=1e-14)
assert checked_mean == float(np.mean(checked_scores))
# Check immediate events, immediate censoring and unchanged follow-up ordering.
zero_control = Surv.from_arrays([True, False, True, False], [0., 0., 1., 2.])
zero_before = zero_control.copy()
zero_labels = xgboost_cox_labels(zero_control)
assert zero_labels[0] == -zero_labels[1] and zero_labels[0] > 0
assert np.array_equal(zero_control, zero_before)
assert np.array_equal(np.argsort(np.abs(zero_labels), kind="stable"), np.argsort(zero_control["time"], kind="stable"))
# Refuse unsupported horizons and invalid probabilities instead of substituting optimistic scores.
try:
    predict_net_survival(synthetic_y, synthetic_margin, synthetic_valid_margin, np.array([5.]))
except ValueError:
    unsupported_rejected = True
else:
    raise AssertionError("An unsupported horizon was accepted.")
try:
    mean_horizon_brier(uncensored_y, uncensored_y, np.full((4, 3), 1.1), uncensored_times)
except ValueError:
    invalid_rejected = True
else:
    raise AssertionError("An invalid probability was accepted.")
show_table(pd.DataFrame({"Control": ["Breslow versus hand calculation: maximum absolute error", "Brier versus direct MSE: maximum absolute error", "Common margin shift", "Bounded, nonincreasing survival", "Invalid probabilities and unsupported horizon"], "Result": [f"{np.max(np.abs(checked_survival - manual_survival)):.3g}", f"{np.max(np.abs(checked_scores - direct_scores)):.3g}", "passed", "passed", "rejected"]}), "Synthetic controls only; not study results")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Control,Result
Breslow versus hand calculation: maximum absolute error,1.11e-16
Brier versus direct MSE: maximum absolute error,0
Common margin shift,passed
"Bounded, nonincreasing survival",passed
Invalid probabilities and unsupported horizon,rejected


Elapsed: 0.000 minutes


## Historical selection criterion
Pareto dominance has no explicit weighting: a candidate is dominated when another has at least as high C and at least as low mean Brier, with one strict improvement. Candidates are subsequently ranked by the original unweighted distance to the ideal point. The following table describes the saved historical candidates; it is not a new performance estimate.

In [5]:
#| label: readm-verify-historical-selection
#| message: false
t0 = time.perf_counter()
# Recompute the selection rule from the original saved final-candidate table.
reference = pd.read_csv(source_path("xgboost_readmission_reference"))
c_column = "Phase2_Multi_Horizon_C_Index"
b_column = "Phase2_Aalen_Johansen_Brier_Score"
assert reference[["trial_id", c_column, b_column, "Distance_to_Ideal"]].notna().all().all()
reference["recomputed_distance"] = distance_to_ideal(reference[c_column], reference[b_column])
assert np.allclose(reference["recomputed_distance"], reference["Distance_to_Ideal"], atol=1e-14)
reference["c_squared_term"] = (1 - reference[c_column])**2
reference["brier_squared_term"] = reference[b_column]**2
reference["c_share_percent"] = 100 * reference["c_squared_term"] / (reference["c_squared_term"] + reference["brier_squared_term"])
reference["rank_distance"] = reference["recomputed_distance"].rank(method="min")
reference["rank_c"] = reference[c_column].rank(ascending=False, method="min")
historical_winner = reference.loc[reference["recomputed_distance"].idxmin()]
assert int(historical_winner["trial_id"]) == int(reference.loc[reference[c_column].idxmax(), "trial_id"])
show_table(reference[["trial_id", c_column, b_column, "c_share_percent", "rank_distance", "rank_c"]].sort_values("rank_distance"), "Saved historical candidates: original mislabeled Brier column retained for traceability")
print("The C share describes squared-distance components; it is not an algorithm weight.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

trial_id,Phase2_Multi_Horizon_C_Index,Phase2_Aalen_Johansen_Brier_Score,c_share_percent,rank_distance,rank_c
0,0.645887,0.110638,91.106435,1.000000,1.000000
33,0.645208,0.110670,91.132832,2.000000,2.000000
23,0.644981,0.110602,91.153073,3.000000,3.000000
10,0.644864,0.110622,91.155373,4.000000,4.000000
43,0.644546,0.110694,91.159342,5.000000,5.000000
17,0.644356,0.110476,91.199699,6.000000,6.000000
4,0.644237,0.110131,91.255052,7.000000,7.000000
6,0.643762,0.109933,91.305016,8.000000,9.000000
11,0.643850,0.110629,91.200309,9.000000,8.000000
9,0.640107,0.109918,91.467862,10.000000,10.000000


The C share describes squared-distance components; it is not an algorithm weight.
Elapsed: 0.000 minutes


## Approved zero-time encoding
Immediate readmissions remain events. A positive numerical representation is used only when constructing the signed XGBoost labels; the analytical follow-up times, baseline-hazard estimation, IPCW, evaluation horizons and exported endpoints retain their original values. The same representation is applied to zero-time events and censoring before assigning the sign, preserving their shared tie.

In [6]:
#| label: readm-audit-zero-time-labels
#| message: false
t0 = time.perf_counter()
# Audit the original signed-time rule using development rows only.
endpoint_columns = ["original_row_id", "is_train", "readmit_event", "readmit_time_from_disch_m"]
development_endpoint = pd.read_parquet(source_path("clean_completed_reference"), columns=endpoint_columns, filters=[("is_train", "=", True)])
assert development_endpoint["is_train"].all() and development_endpoint["original_row_id"].is_unique
assert development_endpoint[endpoint_columns].notna().all().all()
y_development = Surv.from_arrays(development_endpoint["readmit_event"].eq(1), development_endpoint["readmit_time_from_disch_m"])
assert np.isfinite(y_development["time"]).all() and (y_development["time"] >= 0).all()
original_xgb_labels = np.where(y_development["event"], y_development["time"], -y_development["time"]).astype(np.float32)
lost_events = y_development["event"] & ~(original_xgb_labels > 0)
zero_time = y_development["time"] == 0
assert np.array_equal(lost_events, y_development["event"] & zero_time)
label_check = pd.DataFrame({"Check": ["Development participants", "Observed readmissions at time zero", "Censored observations at time zero", "Recorded events lost by signed-time encoding"], "Count": [len(y_development), int((zero_time & y_development["event"]).sum()), int((zero_time & ~y_development["event"]).sum()), int(lost_events.sum())]})
show_table(label_check, "Original label audit; analytical outcomes unchanged")
corrected_xgb_labels = xgboost_cox_labels(y_development)
assert np.array_equal(corrected_xgb_labels > 0, y_development["event"])
assert np.array_equal(np.argsort(np.abs(original_xgb_labels), kind="stable"), np.argsort(np.abs(corrected_xgb_labels), kind="stable"))
assert np.array_equal(original_xgb_labels[~zero_time], corrected_xgb_labels[~zero_time])
print(f"Event mismatches after the approved interface correction: {np.count_nonzero((corrected_xgb_labels > 0) != y_development['event'])}")
print("Analytical times, participants, events and time-zero ties are unchanged.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Check,Count
Development participants,70521
Observed readmissions at time zero,40
Censored observations at time zero,4027
Recorded events lost by signed-time encoding,40


Event mismatches after the approved interface correction: 0
Analytical times, participants, events and time-zero ties are unchanged.
Elapsed: 0.001 minutes


## Completed development and internal partitions
Python reads the five completed Parquet inputs from notebook 00, with an explicit development-only filter. Identity, row order, outcomes, observed programme and absence of missing predictors are checked across imputations. Identifiers, centre metadata and all future outcomes are excluded from predictor encoding.
The initial search retains treatment-plan stratification; the informed search retains treatment-plan by readmission-event stratification. Both use the original seed and five folds. Actual memberships must match the saved reference manifest before any model is fitted.

In [7]:
#| label: readm-completed-data-and-artifacts
#| message: false
t0 = time.perf_counter()
# Use deterministic content fingerprints and dated, reusable artifacts.
object_dir = resolve_path(config["paths"]["objects"])
table_dir = resolve_path(config["paths"]["tables"])
assert object_dir.is_dir() and table_dir.is_dir()
def file_sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=True, allow_nan=False).encode()).hexdigest()[:12]
def artifact_path(prefix, signature, suffix):
    matches = sorted(object_dir.glob(f"{prefix}_????-??-??_{signature}{suffix}"))
    if len(matches) > 1:
        raise RuntimeError(f"Multiple cache artifacts match {prefix} / {signature}.")
    return matches[0] if matches else object_dir / f"{prefix}_{datetime.date.today().isoformat()}_{signature}{suffix}"
def save_pickle(value, path):
    temporary = path.with_suffix(path.suffix + ".partial")
    with temporary.open("wb") as stream:
        pickle.dump(value, stream, protocol=5)
    temporary.replace(path)
    if path not in written:
        written.append(path)
def notebook_code(path, labels):
    notebook = json.loads(path.read_text(encoding="utf-8"))
    selected = {}
    for cell in notebook["cells"]:
        source = "".join(cell.get("source", []))
        for label in labels:
            if source.startswith(f"#| label: {label}\n"):
                if label in selected:
                    raise ValueError(f"Duplicate notebook label: {label}")
                selected[label] = source
    if set(selected) != set(labels):
        raise ValueError("Required notebook code could not be found.")
    return selected
# Load only the five full-development imputations already produced by notebook 00.
completed_paths = [resolve_path(value) for value in config["sources"]["clean_completed_datasets"]]
assert len(completed_paths) == int(config["imputation"]["n_imputations"]) == 5
assert len(set(completed_paths)) == 5 and completed_paths[0] == source_path("clean_completed_reference")
completed_hashes = [file_sha256(path) for path in completed_paths]
metadata_names = ["original_row_id", "hash_key", "legacy_row_id", "is_train", "center_id"]
outcome_names = ["readmit_time_from_disch_m", "readmit_event", "death_time_from_disch_m", "death_event", "readmit_time_from_adm_m", "death_time_from_adm_m"]
development_data = pd.read_parquet(completed_paths[0], filters=[("is_train", "=", True)])
assert development_data["is_train"].all() and development_data["original_row_id"].is_unique
pd.testing.assert_frame_equal(development_data[endpoint_columns], development_endpoint)
predictor_names = [name for name in development_data if name not in metadata_names + outcome_names]
assert len(predictor_names) == 39 and not set(predictor_names).intersection(metadata_names + outcome_names)
historical_exclusions = ["num_trat_ant", "age_subs_onset_rec2", "phys_dx_instudy"]
assert set(historical_exclusions).issubset(predictor_names)
retained_predictors = [name for name in predictor_names if name not in historical_exclusions]
# Keep one completed development dataset in memory; holdout rows never enter model inputs.
@functools.lru_cache(maxsize=1)
def completed_development(imputation):
    if imputation not in range(1, 6):
        raise ValueError("Imputation index must be between one and five.")
    path = completed_paths[imputation - 1]
    if file_sha256(path) != completed_hashes[imputation - 1]:
        raise RuntimeError("A completed input changed during this run.")
    data = pd.read_parquet(path, filters=[("is_train", "=", True)])
    assert list(data.columns) == list(development_data.columns)
    pd.testing.assert_frame_equal(data[metadata_names + outcome_names], development_data[metadata_names + outcome_names])
    assert data["is_train"].all() and data["original_row_id"].is_unique
    assert not data[predictor_names].isna().any().any()
    assert np.array_equal(data["plan_type_corr"], development_data["plan_type_corr"])
    return data
# Reuse the same prepared values in every fold, while fitting encoding on training rows only.
@functools.lru_cache(maxsize=1)
def encoded_fold(scheme, fold, imputation):
    data = completed_development(imputation)
    training, validation = splits[scheme][fold - 1]
    x_train, x_valid = encode_fold(data.iloc[training], data.iloc[validation])
    metadata = {"imputation": imputation, "source": completed_paths[imputation - 1].relative_to(root).as_posix(), "fit_scope": "complete_development_before_cv", "refitted": False}
    return x_train, x_valid, y_development[training], y_development[validation], metadata
input_audit = []
for imputation in range(1, 6):
    completed = completed_development(imputation)
    input_audit.append({"Imputation": imputation, "Development rows": len(completed), "Imputation predictors": len(predictor_names), "Retained ML predictors": len(retained_predictors), "Missing predictors": int(completed[predictor_names].isna().sum().sum()), "Source": completed_paths[imputation - 1].relative_to(root).as_posix()})
del completed
completed_development.cache_clear()
show_table(pd.DataFrame(input_audit), "Verified complete-development imputations from notebook 00")
print("No R subprocess or imputation fit is run here. Metadata, including center_id, never enters the predictor matrix.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Imputation,Development rows,Imputation predictors,Retained ML predictors,Missing predictors,Source
1,70521,39,36,0,outputs/objects/completed_imp1_2026-10-02_71f0ce969d33.parquet
2,70521,39,36,0,outputs/objects/completed_imp2_2026-10-02_71f0ce969d33.parquet
3,70521,39,36,0,outputs/objects/completed_imp3_2026-10-02_71f0ce969d33.parquet
4,70521,39,36,0,outputs/objects/completed_imp4_2026-10-02_71f0ce969d33.parquet
5,70521,39,36,0,outputs/objects/completed_imp5_2026-10-02_71f0ce969d33.parquet


No R subprocess or imputation fit is run here. Metadata, including center_id, never enters the predictor matrix.
Elapsed: 0.020 minutes


In [8]:
#| label: readm-freeze-and-check-folds
#| message: false
t0 = time.perf_counter()
# Preserve the original fold memberships; the imputations are already complete.
plan_codes = development_data["plan_type_corr"].map({"pg-pab": 0, "pg-pr": 1, "m-pr": 2, "pg-pai": 3, "m-pai": 4})
assert plan_codes.notna().all()
strata = {"plan": plan_codes.to_numpy(dtype=int), "dual": plan_codes.to_numpy(dtype=int) * 10 + y_development["event"].astype(int)}
fold_manifest = pd.DataFrame({"original_row_id": development_data["original_row_id"].to_numpy()})
splits = {}
for scheme, labels in strata.items():
    counts = pd.Series(labels).value_counts()
    if len(counts) < 2 or counts.min() < 5:
        raise ValueError("The fixed cohort no longer supports the original stratification.")
    splits[scheme] = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=seed).split(np.zeros(len(labels)), labels))
    membership = np.zeros(len(labels), dtype=np.int8)
    for fold, (training, validation) in enumerate(splits[scheme], start=1):
        assert not np.intersect1d(training, validation).size
        membership[validation] = fold
    assert np.all((membership >= 1) & (membership <= 5))
    fold_manifest[f"{scheme}_fold"] = membership
# Anchor the actual memberships to the previous manifest before any new model fit.
previous_folds = pd.read_parquet(source_path("readmission_fold_reference"))
pd.testing.assert_frame_equal(fold_manifest, previous_folds[fold_manifest.columns])
data_contract = {"imputation_fit_scope": "complete_development_before_cv", "completed_sha256": completed_hashes, "imputation": config["imputation"], "seed": seed, "manifest_sha256": hashlib.sha256(fold_manifest.to_csv(index=False).encode()).hexdigest(), "loader_code": notebook_code(source_path("readmission_notebook"), ["readm-completed-data-and-artifacts"])}
data_signature = fingerprint(data_contract)
manifest_path = artifact_path("xgb_readm_fold_manifest", data_signature, ".parquet")
if manifest_path.exists():
    pd.testing.assert_frame_equal(pd.read_parquet(manifest_path), fold_manifest)
else:
    manifest_path.write_bytes(fold_manifest.to_parquet(index=False))
    pd.testing.assert_frame_equal(pd.read_parquet(manifest_path), fold_manifest)
    written.append(manifest_path)
# Check censoring support at every requested horizon without fitting a survival model.
support_rows = []
for scheme, scheme_splits in splits.items():
    for fold, (training, validation) in enumerate(scheme_splits, start=1):
        y_train, y_valid = y_development[training], y_development[validation]
        censoring = CensoringDistributionEstimator().fit(y_train)
        assert np.all(censoring.predict_proba(horizons) > 0)
        assert np.isfinite(censoring.predict_ipcw(y_valid)).all()
        for horizon in horizons:
            assert np.isfinite(censoring.predict_ipcw(y_valid[y_valid["time"] < horizon])).all()
        assert y_valid["time"].min() <= horizons[0] and horizons[-1] < y_valid["time"].max()
        support_rows.append({"Scheme": scheme, "Fold": fold, "Training": len(training), "Validation": len(validation), "Changed memberships": 0, "IPCW support": "passed"})
show_table(pd.DataFrame(support_rows), "Original internal validation partitions, unchanged")
print("Every CV fold uses values prepared from complete development. Internal CV shares this preprocessing.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Scheme,Fold,Training,Validation,Changed memberships,IPCW support
plan,1,56416,14105,0,passed
plan,2,56417,14104,0,passed
plan,3,56417,14104,0,passed
plan,4,56417,14104,0,passed
plan,5,56417,14104,0,passed
dual,1,56416,14105,0,passed
dual,2,56417,14104,0,passed
dual,3,56417,14104,0,passed
dual,4,56417,14104,0,passed
dual,5,56417,14104,0,passed


Every CV fold uses values prepared from complete development. Internal CV shares this preprocessing.
Elapsed: 0.007 minutes


## Predictor encoding
Ordinal mappings, dummy reference categories and the first-substance grouping follow the original notebook. Category levels and zero-variance removal are determined in each training fold; validation uses that frozen encoding. Unmapped categories raise an error rather than receiving an unreported replacement. Imputation itself has already been completed on all development rows in notebook 00.

In [9]:
#| label: readm-training-only-encoding
#| message: false
t0 = time.perf_counter()
# Preserve the study-defined ordinal scores and dummy references.
clinical_levels = {"logro minimo": 2, "logro intermedio": 1, "logro alto": 0}
ordered_mappings = {name: clinical_levels for name in ["evaluacindelprocesoteraputico", "eva_consumo", "eva_fam", "eva_relinterp", "eva_ocupacion", "eva_sm", "eva_fisica", "eva_transgnorma"]}
ordered_mappings.update({"tenure_status_household": {"illegal settlement": 4, "stays temporarily with a relative": 3, "others": 2, "renting": 1, "owner/transferred dwellings/pays dividends": 0}, "urbanicity_cat": {"1.Rural": 2, "2.Mixed": 1, "3.Urban": 0}, "prim_sub_freq_rec": {"1.\u22641 day/wk": 0, "2.2\u20136 days/wk": 1, "3.Daily": 2}, "ed_attainment_corr": {"3-Completed primary school or less": 2, "2-Completed high school or less": 1, "1-More than high school": 0}})
dummy_reference = {"sex_rec": "man", "plan_type_corr": "pg-pab", "marital_status_rec": "married/cohabiting", "cohabitation": "alone", "sub_dep_icd10_status": "hazardous consumption", "tr_outcome": "completion", "adm_motive": "spontaneous consultation", "tipo_de_vivienda_rec2": "formal housing", "occupation_condition_corr24": "employed", "any_violence": "0.No domestic violence/sex abuse", "first_sub_used": "marijuana", "primary_sub_mod": "marijuana"}
grouped_substances = ["opioids", "others", "hallucinogens", "inhalants", "tranquilizers/hypnotics", "amphetamine-type stimulants"]
def encode_fold(training, validation):
    training, validation = training[retained_predictors].copy(), validation[retained_predictors].copy()
    if training.isna().any().any() or validation.isna().any().any():
        raise ValueError("Model encoding requires completed predictors.")
    for name, mapping in ordered_mappings.items():
        for data in (training, validation):
            values = data[name].astype(str).str.strip()
            if not values.isin(mapping).all():
                raise ValueError(f"Unknown ordinal category: {name}")
            data[name] = values.map(mapping).astype(float)
    for data in (training, validation):
        data["first_sub_used"] = data["first_sub_used"].replace(dict.fromkeys(grouped_substances, "other"))
    for name in training:
        if pd.api.types.is_numeric_dtype(training[name]) or pd.api.types.is_bool_dtype(training[name]):
            continue
        observed = training[name].astype(str).str.strip()
        reference_level = dummy_reference.get(name)
        levels = sorted(observed.unique().tolist())
        if reference_level is not None:
            if reference_level not in levels:
                raise ValueError(f"Missing training reference level: {name}")
            levels = [reference_level] + [level for level in levels if level != reference_level]
        if not validation[name].astype(str).str.strip().isin(levels).all():
            raise ValueError(f"Unseen model-encoding category: {name}")
        for data in (training, validation):
            data[name] = pd.Categorical(data[name].astype(str).str.strip(), categories=levels)
    training = pd.get_dummies(training, drop_first=True, dtype=float)
    validation = pd.get_dummies(validation, drop_first=True, dtype=float)
    keep = training.columns[training.nunique(dropna=False) > 1].tolist()
    import re
    names = [re.sub(r"[^\w]+", "_", name.lower()).strip("_") for name in keep]
    if len(names) != len(set(names)) or not names:
        raise ValueError("Encoded feature names must be unique and nonempty.")
    training, validation = training[keep].astype(np.float32), validation[keep].astype(np.float32)
    training.columns = validation.columns = names
    assert np.isfinite(training.to_numpy()).all() and np.isfinite(validation.to_numpy()).all()
    return training, validation
# Exercise encoding with completed training rows; these checks produce no CV results.
check_training = development_data.iloc[next(iter(splits.values()))[0][0]][retained_predictors]
check_validation = check_training.iloc[:3].copy()
encoded_check, encoded_valid = encode_fold(check_training, check_validation)
changed_validation = check_validation.copy()
changed_validation["adm_age_rec3"] = changed_validation["adm_age_rec3"] + 100
unchanged_training, _ = encode_fold(check_training, changed_validation)
pd.testing.assert_frame_equal(encoded_check, unchanged_training)
bad_validation = check_validation.copy()
bad_validation["ed_attainment_corr"] = "__unknown__"
try:
    encode_fold(check_training, bad_validation)
except ValueError:
    pass
else:
    raise AssertionError("An unknown ordinal category was silently accepted.")
del check_training, check_validation, changed_validation, bad_validation, encoded_check, encoded_valid, unchanged_training
print("Training encoding is invariant to validation changes; unknown ordinal values are rejected.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Training encoding is invariant to validation changes; unknown ordinal values are rejected.
Elapsed: 0.012 minutes


## Reusing completed inputs and model scores
The five completed datasets stay on disk; at most one extra completed dataset and one encoded fold are cached in memory. No fold-specific imputer, OOB table or copied completed matrix is generated. The full-development forests retained by notebook 00 remain available for frozen application.
Data and method fingerprints include the approved imputation scope, completed-input hashes and fixed fold memberships. The same parameters, imputation and fold reuse their saved scores during rescoring. Earlier scores from fold-wise or historical full-cohort imputation cannot match the new contract. Metric errors stop execution rather than becoming C=0.5 or Brier=0.25.

In [10]:
#| label: readm-fold-cache-and-evaluation
#| message: false
t0 = time.perf_counter()
# Score models using the completed-development loader shared with notebook 02.
fixed_xgb = {"objective": "survival:cox", "eval_metric": "cox-nloglik", "tree_method": "hist", "device": "cpu", "nthread": threads, "seed": seed}
initial_ranges = {"learning_rate": ("float", 0.002, 0.035, True), "max_depth": ("int", 4, 9, False), "min_child_weight": ("int", 3, 30, False), "subsample": ("float", 0.60, 0.90, False), "colsample_bytree": ("float", 0.40, 0.70, False), "reg_alpha": ("float", 0.001, 2.0, True), "reg_lambda": ("float", 0.1, 10.0, True), "gamma": ("float", 0.01, 2.0, True)}
refined_ranges = {"learning_rate": ("float", 0.0025, 0.012, True), "max_depth": ("int", 4, 9, False), "min_child_weight": ("int", 4, 30, False), "subsample": ("float", 0.54, 0.92, False), "colsample_bytree": ("float", 0.45, 0.66, False), "reg_alpha": ("float", 0.0001, 2.0, True), "reg_lambda": ("float", 0.1, 3.5, True), "gamma": ("float", 0.2, 2.0, False)}
param_names = list(initial_ranges)
def native_params(values):
    return {name: int(values[name]) if name in ("max_depth", "min_child_weight") else float(values[name]) for name in param_names}
method_labels = ["readm-completed-data-and-artifacts", "readm-freeze-and-check-folds", "readm-predict-net-survival", "readm-encode-zero-time-labels", "readm-training-only-encoding", "readm-fold-cache-and-evaluation", "readm-optuna-and-pareto"]
model_contract = {"data_signature": data_signature, "data_contract": data_contract, "retained_predictors": retained_predictors, "horizons": horizons.tolist(), "probability_target": settings["probability_target"], "brier_aggregation": settings["brier_aggregation"], "zero_time_label_months": settings["zero_time_label_months"], "fixed_xgb": fixed_xgb, "rounds": 1500, "early_stopping": 30, "budgets": {"initial": 100, "refined": 50, "rescore_top_k": [20, 10], "seed_top": 7, "folds": 5, "imputations": 5}, "prediction_trees": "all_returned_trees_as_in_original", "initial_ranges": initial_ranges, "refined_ranges": refined_ranges, "code": notebook_code(source_path("readmission_notebook"), method_labels), "versions": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "xgboost", "optuna", "scikit-learn", "scikit-survival"]}}
model_signature = fingerprint(model_contract)
score_path = artifact_path("xgb_readm_fold_scores", model_signature, ".pkl")
if score_path.exists():
    with score_path.open("rb") as stream:
        score_cache = pickle.load(stream)
    assert score_cache["contract"] == model_contract
else:
    score_cache = {"contract": model_contract, "scores": {}}
new_model_fits = 0
def evaluate_fold(params, scheme, fold, imputation):
    global new_model_fits
    params = native_params(params)
    key = fingerprint({"params": params, "scheme": scheme, "fold": fold, "imputation": imputation})
    if key in score_cache["scores"]:
        return score_cache["scores"][key]
    if mode == "preflight":
        raise RuntimeError("Preflight cannot fit XGBoost models.")
    x_train, x_valid, y_train, y_valid, imputation_metadata = encoded_fold(scheme, fold, imputation)
    model_started = time.perf_counter()
    d_train = xgb.DMatrix(x_train, label=xgboost_cox_labels(y_train), nthread=threads)
    d_valid = xgb.DMatrix(x_valid, label=xgboost_cox_labels(y_valid), nthread=threads)
    booster = xgb.train(dict(fixed_xgb, **params), d_train, num_boost_round=1500, evals=[(d_valid, "validation")], early_stopping_rounds=30, verbose_eval=False)
    fitting_seconds = time.perf_counter() - model_started
    # Preserve the original all-returned-tree prediction policy after early stopping.
    rounds = booster.num_boosted_rounds()
    tree_range = (0, rounds)
    risk_valid = booster.predict(d_valid, iteration_range=tree_range)
    margin_train = booster.predict(d_train, output_margin=True, iteration_range=tree_range)
    margin_valid = booster.predict(d_valid, output_margin=True, iteration_range=tree_range)
    survival = predict_net_survival(y_train, margin_train, margin_valid, horizons)
    c_horizons = [float(concordance_index_ipcw(y_train, y_valid, risk_valid, tau=horizon)[0]) for horizon in horizons]
    global_c = float(concordance_index_ipcw(y_train, y_valid, risk_valid)[0])
    mean_brier, brier_horizons = mean_horizon_brier(y_train, y_valid, survival, horizons)
    if not np.isfinite(c_horizons + [global_c, mean_brier]).all():
        raise ValueError("A fold metric was nonfinite.")
    result = {"params": params, "scheme": scheme, "fold": fold, "imputation": imputation, "C": float(np.mean(c_horizons)), "mean_Brier": mean_brier, "global_C": global_c, "C_by_horizon": c_horizons, "Brier_by_horizon": brier_horizons.tolist(), "best_iteration": int(booster.best_iteration), "returned_rounds": rounds, "features": list(x_train.columns), "fit_seconds": fitting_seconds, "model_and_scoring_seconds": time.perf_counter() - model_started, "imputation_source": imputation_metadata["source"], "imputation_fit_scope": imputation_metadata["fit_scope"], "imputation_refitted": False}
    score_cache["scores"][key] = result
    save_pickle(score_cache, score_path)
    new_model_fits += 1
    del booster, d_train, d_valid, margin_train, margin_valid, risk_valid, survival
    gc.collect()
    return result
show_table(pd.DataFrame({"Parameter": param_names, "Initial_range": [str(initial_ranges[name][1:3]) for name in param_names], "Refined_range": [str(refined_ranges[name][1:3]) for name in param_names]}), "Original search ranges")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Parameter,Initial_range,Refined_range
learning_rate,"(0.002, 0.035)","(0.0025, 0.012)"
max_depth,"(4, 9)","(4, 9)"
min_child_weight,"(3, 30)","(4, 30)"
subsample,"(0.6, 0.9)","(0.54, 0.92)"
colsample_bytree,"(0.4, 0.7)","(0.45, 0.66)"
reg_alpha,"(0.001, 2.0)","(0.0001, 2.0)"
reg_lambda,"(0.1, 10.0)","(0.1, 3.5)"
gamma,"(0.01, 2.0)","(0.2, 2.0)"


Elapsed: 0.000 minutes


## Optuna and all-imputation Pareto rescoring
The initial search uses 100 trials; the informed search uses 50 trials including up to seven configurations from the newly rescored initial candidates. Each trial cycles through one of five imputations; candidate rescoring uses all five imputations and five folds. This assignment is balanced and deterministic, rather than a random imputation draw.
Both samplers now have an explicit seed and studies run sequentially with two native XGBoost threads. The original initial sampler had no explicit seed. The original pruning thresholds and unweighted distance are retained. Informed seed configurations outside its narrower random-search bounds remain eligible, as in the original.
Study and score checkpoints are saved after completed steps. The exploratory grid and the unreported optimism bootstrap are not rerun; their removal is documented.

In [11]:
#| label: readm-optuna-and-pareto
#| message: false
t0 = time.perf_counter()
# Preserve Optuna budgets, candidate limits and pruning; checkpoint sampler state for exact resumption.
def pareto_table(study):
    rows = [{"trial_id": trial.number, "C": trial.values[0], "mean_Brier": trial.values[1], **native_params(trial.params)} for trial in study.best_trials]
    if not rows:
        raise RuntimeError("No complete Pareto candidate is available.")
    table = pd.DataFrame(rows)
    table["Distance_to_Ideal"] = distance_to_ideal(table["C"], table["mean_Brier"])
    return table.sort_values("Distance_to_Ideal", kind="stable").reset_index(drop=True)
def optimize_stage(scheme, trial_budget, seed_configs=()):
    if mode != "full":
        raise RuntimeError("Full tuning requires execution mode full.")
    ranges = initial_ranges if scheme == "plan" else refined_ranges
    minimum_folds, minimum_c = (2, 0.55) if scheme == "plan" else (3, 0.58)
    study_path = artifact_path(f"xgb_readm_optuna_{scheme}", model_signature, ".pkl")
    if study_path.exists():
        with study_path.open("rb") as stream:
            saved = pickle.load(stream)
        assert saved["signature"] == model_signature
        study = saved["study"]
    else:
        study = optuna.create_study(directions=["maximize", "minimize"], sampler=optuna.samplers.NSGAIISampler(seed=seed), study_name=f"readmission_{scheme}_{model_signature}")
        for params in seed_configs:
            study.enqueue_trial(native_params(params))
    if any(trial.state in (optuna.trial.TrialState.FAIL, optuna.trial.TrialState.RUNNING) for trial in study.trials):
        raise RuntimeError("A failed or unfinished trial requires inspection before resuming.")
    def objective(trial):
        params = {}
        for name, (kind, lower, upper, logarithmic) in ranges.items():
            params[name] = trial.suggest_int(name, lower, upper) if kind == "int" else trial.suggest_float(name, lower, upper, log=logarithmic)
        imputation = trial.number % 5 + 1
        trial.set_user_attr("imputation", imputation)
        fold_results = []
        for fold in range(1, 6):
            fold_results.append(evaluate_fold(params, scheme, fold, imputation))
            if len(fold_results) >= minimum_folds and np.mean([row["C"] for row in fold_results]) < minimum_c:
                raise optuna.TrialPruned()
        trial.set_user_attr("global_C", float(np.mean([row["global_C"] for row in fold_results])))
        return float(np.mean([row["C"] for row in fold_results])), float(np.mean([row["mean_Brier"] for row in fold_results]))
    def checkpoint(study, trial):
        save_pickle({"signature": model_signature, "study": study}, study_path)
        print(f"{scheme} trial {trial.number + 1}/{trial_budget}: {trial.state.name}", flush=True)
    finished = sum(trial.state.is_finished() for trial in study.trials)
    if finished > trial_budget:
        raise RuntimeError("Saved study exceeds the fixed trial budget.")
    if finished < trial_budget:
        study.optimize(objective, n_trials=trial_budget - finished, n_jobs=1, callbacks=[checkpoint], show_progress_bar=False)
    assert sum(trial.state.is_finished() for trial in study.trials) == trial_budget
    return study
def rescore_pareto(study, scheme, top_k):
    candidates = pareto_table(study).head(top_k)
    rows, detail = [], {}
    for candidate in candidates.to_dict("records"):
        params = native_params(candidate)
        per_imputation = []
        for imputation in range(1, 6):
            records = [evaluate_fold(params, scheme, fold, imputation) for fold in range(1, 6)]
            per_imputation.append({name: float(np.mean([row[name] for row in records])) for name in ["C", "mean_Brier", "global_C"]})
        trial_id = int(candidate["trial_id"])
        detail[trial_id] = per_imputation
        rows.append({"trial_id": trial_id, **{name: float(np.mean([row[name] for row in per_imputation])) for name in ["C", "mean_Brier", "global_C"]}, "C_SD_across_imputations": float(np.std([row["C"] for row in per_imputation])), "Brier_SD_across_imputations": float(np.std([row["mean_Brier"] for row in per_imputation])), **params})
    table = pd.DataFrame(rows)
    table["Distance_to_Ideal"] = distance_to_ideal(table["C"], table["mean_Brier"])
    table = table.sort_values("Distance_to_Ideal", kind="stable").reset_index(drop=True)
    path = artifact_path(f"xgb_readm_rescored_{scheme}", model_signature, ".pkl")
    save_pickle({"signature": model_signature, "table": table, "detail": detail}, path)
    return table
def informed_seeds(table):
    configs, seen = [], set()
    for row in table.head(7).to_dict("records"):
        params = native_params(row)
        key = json.dumps(params, sort_keys=True)
        if key not in seen:
            configs.append(params)
            seen.add(key)
    return configs
print("Two-objective search functions ready. No study is started in preflight.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Two-objective search functions ready. No study is started in preflight.
Elapsed: 0.000 minutes


## Execute the authorized stage
Preflight checks every completed-data CV encoding and writes only the fixed fold manifest. Pilot mode measures one model fit and its scoring using the historical winner; it fits no imputer and supplies no reportable selection result. Full mode executes both new searches and their all-imputation rescoring.
Historical results are comparison references only. Relative to the manuscript workflow, the imputation scope, probabilities, zero-time encoding and initial sampler reproducibility changed together; their separate contributions cannot be inferred from the aggregate comparison.

In [12]:
#| label: readm-run-authorized-stage
#| message: false
t0 = time.perf_counter()
# Execute only the compute stage selected after the user's resource decision.
pilot_result, initial_table, final_table = None, None, None
if mode == "preflight":
    # Check every actual fold/imputation encoding before authorizing expensive model fits.
    encoding_audit = []
    for imputation in range(1, 6):
        for scheme in splits:
            for fold in range(1, 6):
                x_train, x_valid, y_train, y_valid, metadata = encoded_fold(scheme, fold, imputation)
                assert len(x_train) + len(x_valid) == len(development_data)
                assert not set(metadata_names + outcome_names).intersection(x_train.columns)
                encoding_audit.append({"Imputation": imputation, "Scheme": scheme, "Fold": fold, "Encoded predictors": x_train.shape[1], "Missing training": int(x_train.isna().sum().sum()), "Missing validation": int(x_valid.isna().sum().sum())})
    show_table(pd.DataFrame(encoding_audit), "All completed-data CV encodings verified without model fitting")
    del x_train, x_valid, y_train, y_valid, metadata
    print("Preflight passed. No imputer, XGBoost model or Optuna trial was fitted.")
elif mode == "pilot":
    pilot_result = evaluate_fold(native_params(historical_winner), "plan", 1, 1)
    show_table(pd.DataFrame({"Item": ["Purpose", "New imputation fits", "Model fitting time (minutes)", "Model plus scoring time (minutes)", "Returned boosting rounds", "Best iteration", "Encoded predictors"], "Value": ["Timing only; not reportable", 0, pilot_result["fit_seconds"] / 60, pilot_result["model_and_scoring_seconds"] / 60, pilot_result["returned_rounds"], pilot_result["best_iteration"], len(pilot_result["features"])]}), "Single-fold timing pilot")
else:
    initial_study = optimize_stage("plan", 100)
    show_table(pareto_table(initial_study), "Initial Pareto candidates, one imputation per trial")
    initial_table = rescore_pareto(initial_study, "plan", 20)
    show_table(initial_table, "Initial candidates rescored on all five imputations")
    refined_study = optimize_stage("dual", 50, informed_seeds(initial_table))
    show_table(pareto_table(refined_study), "Informed Pareto candidates, one imputation per trial")
    final_table = rescore_pareto(refined_study, "dual", 10)
    show_table(final_table, "Final candidates rescored on all five imputations")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

[I 2026-10-03 12:42:36,461] A new study created in memory with name: readmission_plan_f53816455a8a


[I 2026-10-03 12:43:25,987] Trial 0 finished with values: [0.6457070806524315, 0.10946563111746768] and parameters: {'learning_rate': 0.0038280102753304847, 'max_depth': 7, 'min_child_weight': 3, 'subsample': 0.6395345367575241, 'colsample_bytree': 0.6442393742200758, 'reg_alpha': 0.1436757127036174, 'reg_lambda': 0.7578459397323387, 'gamma': 0.0829340446620956}.


plan trial 1/100: COMPLETE


[I 2026-10-03 12:44:05,283] Trial 1 finished with values: [0.6449645858055539, 0.1095862755102833] and parameters: {'learning_rate': 0.008582013741066118, 'max_depth': 9, 'min_child_weight': 25, 'subsample': 0.723327121538445, 'colsample_bytree': 0.5644318856745857, 'reg_alpha': 0.8237691529375192, 'reg_lambda': 3.1720500897865564, 'gamma': 0.2295819372628562}.


plan trial 2/100: COMPLETE


...

[I 2026-10-03 14:31:30,791] Trial 49 finished with values: [0.6439123806325198, 0.1096096157049364] and parameters: {'learning_rate': 0.008981872361916741, 'max_depth': 9, 'min_child_weight': 26, 'subsample': 0.783538217200631, 'colsample_bytree': 0.5299423719481829, 'reg_alpha': 0.009430802008159196, 'reg_lambda': 0.12591702328365986, 'gamma': 1.265525200339976}.


dual trial 50/50: COMPLETE


trial_id,C,mean_Brier,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,Distance_to_Ideal
0,0.645976,0.109520,0.003938,9,10,0.827216,0.410145,0.001269,0.125946,0.036363,0.370578
35,0.645266,0.109516,0.004615,8,13,0.837387,0.503877,0.078659,2.456216,1.662751,0.371254
4,0.645084,0.109469,0.003568,7,4,0.590077,0.620968,0.064704,0.477602,0.918686,0.371415
13,0.644334,0.109465,0.006842,6,16,0.882062,0.584212,0.002336,1.348071,0.977658,0.372130
29,0.644105,0.109459,0.009755,4,8,0.696513,0.633869,0.002135,0.181507,1.470188,0.372347
19,0.644074,0.109423,0.003921,6,8,0.649006,0.562403,0.126984,0.526544,1.658138,0.372367


trial_id,C,mean_Brier,global_C,C_SD_across_imputations,Brier_SD_across_imputations,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,Distance_to_Ideal
0,0.646072,0.109508,0.616199,0.000150,0.000012,0.003938,9,10,0.827216,0.410145,0.001269,0.125946,0.036363,0.370482
35,0.645193,0.109516,0.616962,0.000050,0.000011,0.004615,8,13,0.837387,0.503877,0.078659,2.456216,1.662751,0.371324
4,0.644907,0.109470,0.616286,0.000122,0.000009,0.003568,7,4,0.590077,0.620968,0.064704,0.477602,0.918686,0.371584
13,0.644822,0.109494,0.616767,0.000266,0.000018,0.006842,6,16,0.882062,0.584212,0.002336,1.348071,0.977658,0.371672
19,0.643977,0.109421,0.616246,0.000100,0.000009,0.003921,6,8,0.649006,0.562403,0.126984,0.526544,1.658138,0.372459
29,0.643859,0.109468,0.616795,0.000147,0.000016,0.009755,4,8,0.696513,0.633869,0.002135,0.181507,1.470188,0.372585


Elapsed: 127.068 minutes


## Manuscript comparison and handoff
The final hyperparameter table contributes the XGBoost readmission entries of eTable 2. Its values are compared at the four decimals reported for continuous parameters, with integer depth and child weight. A changed published entry requires the user's checkpoint before proceeding to notebook 02.
Brier comparisons describe different probability calculations: the historical approximation and corrected net survival. They are not an isolated leakage-effect estimate. The complete held-out comparison required by the manuscript remains a later notebook.

In [13]:
#| label: readm-export-winner-and-comparison
#| message: false
t0 = time.perf_counter()
# Export only the final manuscript-facing hyperparameters and a reusable model-selection bundle.
parameter_comparison = pd.DataFrame()
if final_table is not None:
    winner = final_table.iloc[0]
    winner_params = native_params(winner)
    bundle_path = artifact_path("xgb_readm_winner", model_signature, ".pkl")
    parameter_comparison = pd.DataFrame([{"Parameter": name, "Historical": native_params(historical_winner)[name], "Development_pipeline": winner_params[name], "Difference": winner_params[name] - native_params(historical_winner)[name], "Reported_decimals": 0 if name in ("max_depth", "min_child_weight") else 4, "Classification": "substantial" if round(winner_params[name], 0 if name in ("max_depth", "min_child_weight") else 4) != round(native_params(historical_winner)[name], 0 if name in ("max_depth", "min_child_weight") else 4) else "unchanged_at_reported_precision"} for name in param_names])
    summary = {"trial_id": int(winner["trial_id"]), **{name: float(winner[name]) for name in ["C", "mean_Brier", "global_C", "Distance_to_Ideal"]}}
    save_pickle({"signature": model_signature, "contract": model_contract, "params": winner_params, "summary": summary, "comparison": parameter_comparison, "final_candidates": final_table, "manuscript_checkpoint_required": True}, bundle_path)
    artifact_date = bundle_path.name.removeprefix("xgb_readm_winner_").split("_")[0]
    table_path = table_dir / f"etable2_xgboost_readmission_{artifact_date}_{model_signature}.csv"
    manuscript_table = pd.DataFrame([{"Outcome": "Readmission", "Parameter": name, "Initial_search_lower": initial_ranges[name][1], "Initial_search_upper": initial_ranges[name][2], "Search_scale": "log" if initial_ranges[name][3] else "linear", "Selected_value": winner_params[name], "Trial": int(winner["trial_id"]), "Source_cell": "readm-export-winner-and-comparison"} for name in param_names])
    if table_path.exists():
        pd.testing.assert_frame_equal(pd.read_csv(table_path), manuscript_table, check_dtype=False)
    else:
        manuscript_table.to_csv(table_path, index=False)
        written.append(table_path)
    show_table(parameter_comparison, "eTable 2: historical versus development-imputed hyperparameters")
    metric_comparison = pd.DataFrame({"Metric": ["Mean horizon Uno C", "Mean horizon Brier", "Global Uno C"], "Historical": [float(historical_winner[c_column]), float(historical_winner[b_column]), float(historical_winner["Phase2_Global_C_Index"])], "Development_pipeline": [summary["C"], summary["mean_Brier"], summary["global_C"]]})
    metric_comparison["Difference"] = metric_comparison["Development_pipeline"] - metric_comparison["Historical"]
    show_table(metric_comparison, "Saved historical tuning references versus new final selection")
    # Keep the superseded fold-imputation run visible as a separate methodological reference.
    with source_path("xgboost_readmission_superseded_winner").open("rb") as stream:
        previous_fold_run = pickle.load(stream)
    assert previous_fold_run["signature"] != model_signature
    scope_comparison = pd.DataFrame({"Metric": ["Mean horizon Uno C", "Mean horizon Brier", "Global Uno C"], "Previous_fold_imputation": [previous_fold_run["summary"][name] for name in ["C", "mean_Brier", "global_C"]], "Development_imputation": [summary[name] for name in ["C", "mean_Brier", "global_C"]]})
    scope_comparison["Difference"] = scope_comparison["Development_imputation"] - scope_comparison["Previous_fold_imputation"]
    show_table(scope_comparison, "Imputation-scope change: superseded fold-wise run versus current development-imputed tuning")
    print("Review changed manuscript entries before adopting this winner. The user authorized both tuning notebooks; notebook 03 requires a checkpoint.")
else:
    print("Final eTable 2 entries and winner bundle are pending full tuning; no pilot metric is exported as a manuscript result.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Parameter,Historical,Development_pipeline,Difference,Reported_decimals,Classification
learning_rate,0.003377,0.003938,0.000561,4,substantial
max_depth,9.000000,9.000000,0.000000,0,unchanged_at_reported_precision
min_child_weight,4.000000,10.000000,6.000000,0,substantial
subsample,0.610275,0.827216,0.216941,4,substantial
colsample_bytree,0.405147,0.410145,0.004998,4,substantial
reg_alpha,0.284944,0.001269,-0.283675,4,substantial
reg_lambda,0.668159,0.125946,-0.542213,4,substantial
gamma,0.043185,0.036363,-0.006822,4,substantial


Metric,Historical,Development_pipeline,Difference
Mean horizon Uno C,0.645887,0.646072,0.000185
Mean horizon Brier,0.110638,0.109508,-0.001130
Global Uno C,0.615911,0.616199,0.000288


Metric,Previous_fold_imputation,Development_imputation,Difference
Mean horizon Uno C,0.645939,0.646072,0.000133
Mean horizon Brier,0.109552,0.109508,-0.000044
Global Uno C,0.616274,0.616199,-0.000075


Review changed manuscript entries before adopting this winner. The user authorized both tuning notebooks; notebook 03 requires a checkpoint.
Elapsed: 0.002 minutes


## Interpreting the hyperparameter changes
This section uses the selected parameters and cached final fold evaluations. It performs no additional model fitting.

In [14]:
#| label: readm-interpret-hyperparameter-changes
#| message: false
t0 = time.perf_counter()
# Interpret parameter directions without changing the selected model or fitting new models.
if final_table is None:
    print("Interpretation is pending full tuning or reuse of its complete saved results.")
else:
    direction_notes = {
        "learning_rate": ("Larger contribution from each new tree, holding other settings fixed.", "Smaller contribution from each new tree, holding other settings fixed."),
        "max_depth": ("A greater maximum tree depth is allowed.", "A smaller maximum tree depth is allowed."),
        "min_child_weight": ("More restrictive threshold for splitting a node.", "Less restrictive threshold for splitting a node."),
        "subsample": ("More training rows are sampled for each tree.", "Fewer training rows are sampled for each tree."),
        "colsample_bytree": ("More predictor columns are available to each tree.", "Fewer predictor columns are available to each tree."),
        "reg_alpha": ("Stronger L1 penalty on leaf weights.", "Weaker L1 penalty on leaf weights."),
        "reg_lambda": ("Stronger L2 penalty on leaf weights.", "Weaker L2 penalty on leaf weights."),
        "gamma": ("A larger loss reduction is required before splitting.", "A smaller loss reduction is sufficient before splitting.")}
    interpretation = parameter_comparison[["Parameter", "Historical", "Development_pipeline", "Difference", "Classification"]].copy()
    interpretation["Relative_change_percent"] = 100 * interpretation["Difference"] / interpretation["Historical"]
    interpretation["Interpretation"] = ["Unchanged." if row.Difference == 0 else direction_notes[row.Parameter][0 if row.Difference > 0 else 1] for row in interpretation.itertuples()]
    interpretation = interpretation.rename(columns={"Historical": "Previous", "Development_pipeline": "Updated", "Relative_change_percent": "Relative change (%)"})
    show_table(interpretation[["Parameter", "Previous", "Updated", "Relative change (%)", "Interpretation", "Classification"]], "Direction and magnitude of the selected hyperparameter changes")
    # Describe the observed stopping behavior of the winning configuration using cached final evaluations.
    selected_runs = [row for row in score_cache["scores"].values() if row["scheme"] == "dual" and row["params"] == winner_params]
    assert len(selected_runs) == model_contract["budgets"]["folds"] * model_contract["budgets"]["imputations"]
    rounds = np.asarray([row["returned_rounds"] for row in selected_runs])
    show_table(pd.DataFrame([{"Final fold/imputation evaluations": len(selected_runs), "Minimum returned rounds": int(rounds.min()), "Median returned rounds": float(np.median(rounds)), "Maximum returned rounds": int(rounds.max()), "Configured round limit": model_contract["rounds"], "Evaluations reaching the limit": int((rounds == model_contract["rounds"]).sum())}]), "Observed stopping behavior of the selected configuration")
    print("Relative parameter changes are not percentage changes in training speed, model complexity or predictive performance.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Parameter,Previous,Updated,Relative change (%),Interpretation,Classification
learning_rate,0.003377,0.003938,16.612647,"Larger contribution from each new tree, holding other settings fixed.",substantial
max_depth,9.000000,9.000000,0.000000,Unchanged.,unchanged_at_reported_precision
min_child_weight,4.000000,10.000000,150.000000,More restrictive threshold for splitting a node.,substantial
subsample,0.610275,0.827216,35.548098,More training rows are sampled for each tree.,substantial
colsample_bytree,0.405147,0.410145,1.233640,More predictor columns are available to each tree.,substantial
reg_alpha,0.284944,0.001269,-99.554652,Weaker L1 penalty on leaf weights.,substantial
reg_lambda,0.668159,0.125946,-81.150295,Weaker L2 penalty on leaf weights.,substantial
gamma,0.043185,0.036363,-15.797658,A smaller loss reduction is sufficient before splitting.,substantial


Final fold/imputation evaluations,Minimum returned rounds,Median returned rounds,Maximum returned rounds,Configured round limit,Evaluations reaching the limit
25,1151,1330.000000,1500,1500,3


Relative parameter changes are not percentage changes in training speed, model complexity or predictive performance.
Elapsed: 0.000 minutes


The learning-rate change describes the contribution of a new tree, considered in isolation. It does not establish the convergence speed of the full model. Regularization, sampling, depth and child-weight changes act through different controls and cannot be added into one measure of complexity. Definitions follow the [official XGBoost parameter documentation](https://xgboost.readthedocs.io/en/release_3.1.0/parameter.html#parameters-for-tree-booster).
The stopping summary describes the new configuration only. Historical and updated learning curves have not been compared; stopping below the round limit is not proof of absence of overfitting.
Here, **substantial** means that a parameter changes at the reported precision of eTable 2. It does not mean substantial deterioration in prediction. Tuning comparisons do not establish stable SHAP rules, calibration or held-out performance.
The earlier fold-imputation winner was accepted before the investigator chose complete-development imputation. That earlier acceptance does not transfer to this new winner. Both tuning notebooks are authorized to run; changed manuscript entries require review before adoption and before notebook 03.
### Complete-development rerun checkpoint (2026-10-03)
The full run reused the five completed development datasets from notebook 00. It made no new imputation fits. The original participant split, CV memberships, search settings, selection rule and scoring implementation were retained. The superseded fold-imputation run is a separate historical comparator, not the manuscript baseline.
| Parameter | Superseded fold-imputation winner | Current complete-development winner | Classification |
|---|---:|---:|---|
| min_child_weight | 5 | 10 | Substantial: a different selected configuration |
| colsample_bytree | 0.468497 | 0.410145 | Substantial: a different selected configuration |
The learning rate and the other five parameters are identical to the superseded winner. The selected configuration changed, so adopting it for SHAP requires the investigator's checkpoint. Seven parameters still differ from the original manuscript table; the full manuscript comparison is displayed above.
| Internal tuning metric | Superseded fold-imputation run | Complete-development run | Difference |
|---|---:|---:|---:|
| C | 0.645939 | 0.646072 | +0.000133 |
| mean_Brier | 0.109552 | 0.109508 | -0.000044 |
| global_C | 0.616274 | 0.616199 | -0.000075 |
The tuning metrics remain close descriptively. These are comparisons of two selected winners, not a paired experiment with fixed parameters, an equivalence test or a held-out performance estimate. They do not establish stability of the Cox specifications, Tables 2-4 or eTables 11-15.
Imputation is shared across internal CV folds; this CV does not independently evaluate the whole preprocessing procedure. Holdout rows did not fit the imputers or tune this model. No claim of complete independence of historical discovery decisions is made.
Investigator decision: option A was accepted on 2026-10-03. Adopt this complete-development winner and proceed with the full mortality tuning in notebook 02. The additional fixed-parameter comparison was not requested. The original manuscript remains unchanged; notebook 03 and its SHAP work require the next checkpoint.


## Resources and execution status

In [15]:
#| label: readm-resource-and-status
#| message: false
t0 = time.perf_counter()
# Report current resources; the external runner records the sampled process-tree peak.
encoded_fold.cache_clear()
completed_development.cache_clear()
gc.collect()
show_table(pd.DataFrame({"Item": ["Mode", "Preflight controls", "New cohort model fits in this execution", "Full tuning complete", "Current Python RSS (GiB)", "Available RAM (GiB)", "Native threads", "Elapsed after setup (minutes)"], "Value": [mode, "passed", new_model_fits, final_table is not None, round(process.memory_info().rss / 2**30, 3), round(psutil.virtual_memory().available / 2**30, 3), threads, round((time.perf_counter() - run_started) / 60, 3)]}), "Execution status")
if mode == "preflight":
    print("Preflight estimates no tuning performance. Full mode runs the authorized search after its resource review.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Item,Value
Mode,full
Preflight controls,passed
New cohort model fits in this execution,945
Full tuning complete,True
Current Python RSS (GiB),0.851000
Available RAM (GiB),56.350000
Native threads,2
Elapsed after setup (minutes),127.166000


Elapsed: 0.001 minutes


## Session info {#readm-execution-session}

In [16]:
#| label: readm-session-info
#| message: false
t0 = time.perf_counter()
# Record the actual Python environment and native thread limits.
package_names = ["numpy", "pandas", "xgboost", "optuna", "scikit-learn", "scikit-survival", "pyarrow", "PyYAML", "psutil", "threadpoolctl"]
show_table(pd.DataFrame({"Package": package_names, "Version": [importlib.metadata.version(name) for name in package_names]}), "Installed execution packages")
show_table(pd.DataFrame([{"Library": row.get("internal_api"), "Threads": row.get("num_threads"), "Version": row.get("version")} for row in threadpool_info()]), "Native thread pools")
assert all(row["num_threads"] <= threads for row in threadpool_info())
print("Imputation provenance: notebook 00 used R/r-main. This tuning session uses Python/surv only.")
print(f"Python {platform.python_version()} | {platform.system()} {platform.release()} | seed {seed}")
print(f"Elapsed since setup: {(time.perf_counter() - run_started) / 60:.3f} minutes")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Package,Version
numpy,2.0.1
pandas,3.0.0
xgboost,3.1.3
optuna,4.7.0
scikit-learn,1.8.0
scikit-survival,0.27.0
pyarrow,15.0.2
PyYAML,6.0.3
psutil,7.2.2
threadpoolctl,3.6.0


Library,Threads,Version
mkl,2,2023.1-Product
openmp,2,NaN
openmp,2,NaN


Imputation provenance: notebook 00 used R/r-main. This tuning session uses Python/surv only.
Python 3.11.14 | Windows 10 | seed 2125
Elapsed since setup: 127.172 minutes
Elapsed: 0.001 minutes


### Files written and execution status

In [17]:
#| label: readm-written-files-and-status
#| message: false
t0 = time.perf_counter()
# List only artifacts actually written or updated by this execution.
manifest = pd.DataFrame([{"Path": str(path.relative_to(root)).replace("\\", "/"), "Bytes": path.stat().st_size} for path in dict.fromkeys(written)], columns=["Path", "Bytes"])
show_table(manifest, "Files written in outputs")
execution_status = {"mode": mode, "imputation_fit_scope": "complete_development_before_cv", "new_imputation_fits": 0, "full_tuning_complete": final_table is not None, "new_model_fits": new_model_fits, "data_signature": data_signature, "model_signature": model_signature, "checkpoint_before_notebook_02": True}
print(json.dumps(execution_status, sort_keys=True))
print("Full tuning finished; review the comparison before notebook 02." if final_table is not None else "Preflight/pilot finished; full tuning and the notebook 01 checkpoint remain pending.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Path,Bytes
outputs/objects/xgb_readm_fold_scores_2026-10-03_f53816455a8a.pkl,1696149
outputs/objects/xgb_readm_optuna_plan_2026-10-03_f53816455a8a.pkl,64034
outputs/objects/xgb_readm_rescored_plan_2026-10-03_f53816455a8a.pkl,2977
outputs/objects/xgb_readm_optuna_dual_2026-10-03_f53816455a8a.pkl,38854
outputs/objects/xgb_readm_rescored_dual_2026-10-03_f53816455a8a.pkl,3288
outputs/objects/xgb_readm_winner_2026-10-03_f53816455a8a.pkl,40769
outputs/tables/etable2_xgboost_readmission_2026-10-03_f53816455a8a.csv,849


{"checkpoint_before_notebook_02": true, "data_signature": "93a86001d458", "full_tuning_complete": true, "imputation_fit_scope": "complete_development_before_cv", "mode": "full", "model_signature": "f53816455a8a", "new_imputation_fits": 0, "new_model_fits": 945}
Full tuning finished; review the comparison before notebook 02.
Elapsed: 0.001 minutes
